# Dataclass — Nível Avançado

**Objetivo:** usar a introspecção das dataclasses para construir ferramentas (loader e validador genéricos),
criar classes dinamicamente, escrever decoradores próprios compatíveis com type checkers, medir memória/tempo
de verdade e desenhar a fronteira entre dataclass (núcleo) e Pydantic (borda).

Roteiro:

1. Loader genérico `dict → dataclass` guiado por type hints (um "mini-Pydantic")
2. Validação declarativa via `metadata`
3. `make_dataclass`: classes geradas em runtime
4. Dataclasses genéricas: `Result[T]` para serviços
5. Seu próprio decorador com `dataclass_transform` (PEP 681)
6. `cached_property`, `slots` e `frozen`: o que combina com o quê
7. `copy.replace` e `__replace__` (3.13)
8. Benchmark: memória e tempo de criação
9. Ponte com Pydantic: validar **para dentro** de uma dataclass stdlib
10. Arquitetura de bordas: DTO Pydantic ↔ domínio dataclass
11. Checklist

In [1]:
import sys, json, timeit, tracemalloc, types, typing
from dataclasses import dataclass, field, fields, is_dataclass, make_dataclass, asdict, MISSING
from datetime import datetime, date, timezone
from decimal import Decimal
from enum import Enum
from typing import Any, Union, get_args, get_origin, get_type_hints
from uuid import UUID
print(sys.version.split()[0])

3.13.18


## 1. Loader genérico guiado por type hints

No intermediário, cada classe precisou de um `from_json` manual. Como as dataclasses expõem `fields()` e os type hints
estão disponíveis via `typing.get_type_hints`, dá para escrever **um** conversor recursivo que serve para qualquer dataclass.

Construir isso uma vez é o melhor jeito de entender o que o Pydantic faz por baixo (e por que não vale a pena manter o seu em produção).

In [2]:
class LoadError(ValueError):
    def __init__(self, path: str, msg: str):
        super().__init__(f"{path or '<root>'}: {msg}")
        self.path = path

_SCALARS = {
    int: lambda v: v if isinstance(v, int) and not isinstance(v, bool) else int(str(v)),
    float: float,
    str: lambda v: v if isinstance(v, str) else (_ for _ in ()).throw(TypeError(f"esperado str, recebido {type(v).__name__}")),
    bool: lambda v: v if isinstance(v, bool) else {"true": True, "false": False, "1": True, "0": False}[str(v).lower()],
    Decimal: lambda v: Decimal(str(v)),
    UUID: lambda v: v if isinstance(v, UUID) else UUID(str(v)),
    datetime: lambda v: v if isinstance(v, datetime) else datetime.fromisoformat(v),
    date: lambda v: v if isinstance(v, date) else date.fromisoformat(v),
}

def load(tp: Any, value: Any, path: str = "") -> Any:
    origin, args = get_origin(tp), get_args(tp)

    if tp is Any:
        return value
    if origin in (Union, types.UnionType):                   # Optional[X], X | Y
        if value is None and type(None) in args:
            return None
        errors = []
        for arg in (a for a in args if a is not type(None)):
            try:
                return load(arg, value, path)
            except (LoadError, ValueError, TypeError, KeyError) as e:
                errors.append(str(e))
        raise LoadError(path, f"nenhum tipo da união aceitou {value!r}")
    if origin is typing.Literal:
        if value not in args:
            raise LoadError(path, f"esperado um de {args}, recebido {value!r}")
        return value
    if origin in (list, set, frozenset, tuple):
        if not isinstance(value, (list, tuple, set)):
            raise LoadError(path, f"esperado lista, recebido {type(value).__name__}")
        if origin is tuple and not (len(args) == 2 and args[1] is Ellipsis):
            return tuple(load(a, v, f"{path}[{i}]") for i, (a, v) in enumerate(zip(args, value)))
        item_tp = args[0] if args else Any
        return origin(load(item_tp, v, f"{path}[{i}]") for i, v in enumerate(value))
    if origin is dict:
        k_tp, v_tp = args or (Any, Any)
        return {load(k_tp, k, f"{path}.<key>"): load(v_tp, v, f"{path}.{k}") for k, v in value.items()}
    if isinstance(tp, type) and issubclass(tp, Enum):
        try:
            return tp(value)
        except ValueError:
            raise LoadError(path, f"valor {value!r} não pertence a {tp.__name__}") from None
    if is_dataclass(tp):
        if isinstance(value, tp):
            return value
        if not isinstance(value, dict):
            raise LoadError(path, f"esperado objeto, recebido {type(value).__name__}")
        hints = get_type_hints(tp)
        kwargs = {}
        for f in fields(tp):
            if not f.init:
                continue
            sub = f"{path}.{f.name}" if path else f.name
            if f.name in value:
                kwargs[f.name] = load(hints[f.name], value[f.name], sub)
            elif f.default is MISSING and f.default_factory is MISSING:
                raise LoadError(sub, "campo obrigatório ausente")
        unknown = set(value) - {f.name for f in fields(tp)}
        if unknown:
            raise LoadError(path, f"campos desconhecidos: {sorted(unknown)}")
        return tp(**kwargs)
    if tp in _SCALARS:
        try:
            return _SCALARS[tp](value)
        except (ValueError, TypeError, KeyError) as e:
            raise LoadError(path, f"não converte {value!r} para {tp.__name__} ({e})") from None
    raise LoadError(path, f"tipo não suportado: {tp!r}")

In [3]:
class Status(Enum):
    OPEN = "open"
    PAID = "paid"

@dataclass(frozen=True)
class Item:
    sku: str
    qty: int
    unit_price: Decimal

@dataclass(frozen=True)
class Order:
    id: UUID
    status: Status
    created_at: datetime
    items: tuple[Item, ...]
    discount: Decimal | None = None
    metadata: dict[str, str] = field(default_factory=dict)

raw = json.loads("""{
  "id": "0b8e6c6e-52a5-4b8f-9d1e-3c1f8a7e2b10",
  "status": "paid",
  "created_at": "2026-11-10T09:30:00+00:00",
  "items": [{"sku": "A1", "qty": "2", "unit_price": "19.90"}, {"sku": "B2", "qty": 1, "unit_price": 100}],
  "metadata": {"channel": "app"}
}""")
order = load(Order, raw)
print(order)

Order(id=UUID('0b8e6c6e-52a5-4b8f-9d1e-3c1f8a7e2b10'), status=<Status.PAID: 'paid'>, created_at=datetime.datetime(2026, 11, 10, 9, 30, tzinfo=datetime.timezone.utc), items=(Item(sku='A1', qty=2, unit_price=Decimal('19.90')), Item(sku='B2', qty=1, unit_price=Decimal('100'))), discount=None, metadata={'channel': 'app'})


In [4]:
bad = json.loads(json.dumps(raw))
bad["items"][1]["qty"] = "um"
bad["status"] = "cancelled"
for case in ({**raw, "items": bad["items"]}, {**raw, "status": "cancelled"}, {**raw, "coupon": "X"}):
    try:
        load(Order, case)
    except LoadError as e:
        print("LoadError:", e)

LoadError: items[1].qty: não converte 'um' para int (invalid literal for int() with base 10: 'um')
LoadError: status: valor 'cancelled' não pertence a Status
LoadError: <root>: campos desconhecidos: ['coupon']


~90 linhas e já temos: conversão, aninhamento, uniões, enums, coleções e caminho do erro. **Faltam**: coletar *todos* os erros
de uma vez, strict/lax configurável, restrições (`gt`, `max_length`), aliases, serialização, JSON Schema, performance (isso é Python puro)
e anos de casos de borda. Esse é o argumento honesto para usar Pydantic na borda — e a seção 9 mostra que ele faz exatamente
isso **com a mesma dataclass**, sem mudá-la.

## 2. Validação declarativa via `metadata`

Combinando `metadata` com um decorador, dá para ter restrições declarativas e checagem de tipo **opcional** por classe —
útil em bibliotecas que não podem depender do Pydantic.

In [5]:
def check(**rules):
    """Atalho: field(metadata={'rules': {...}})"""
    default = rules.pop("default", MISSING)
    return field(default=default, metadata={"rules": rules})

def _isinstance_shallow(value, tp) -> bool:
    origin = get_origin(tp) or tp
    if origin in (Union, types.UnionType):
        return any(_isinstance_shallow(value, a) for a in get_args(tp))
    if tp is Any:
        return True
    if tp is float:
        return isinstance(value, (int, float)) and not isinstance(value, bool)
    return isinstance(value, origin)

def validated(cls):
    """Adiciona checagem de tipo (rasa) + regras do metadata ao __post_init__, preservando o original.

    Deve ser aplicado ANTES do @dataclass (ou seja, por dentro): o __init__ gerado só chama
    __post_init__ se o método já existir no momento em que o @dataclass roda.
    """
    original = cls.__dict__.get("__post_init__")

    def __post_init__(self, *args):
        hints = get_type_hints(type(self))
        errors = []
        for f in fields(self):
            v = getattr(self, f.name)
            if not _isinstance_shallow(v, hints[f.name]):
                errors.append(f"{f.name}: esperado {hints[f.name]}, recebido {type(v).__name__}")
                continue
            r = f.metadata.get("rules", {})
            if "ge" in r and v < r["ge"]:
                errors.append(f"{f.name}: deve ser >= {r['ge']}")
            if "le" in r and v > r["le"]:
                errors.append(f"{f.name}: deve ser <= {r['le']}")
            if "max_len" in r and len(v) > r["max_len"]:
                errors.append(f"{f.name}: tamanho máximo {r['max_len']}")
            if "choices" in r and v not in r["choices"]:
                errors.append(f"{f.name}: deve ser um de {r['choices']}")
        if errors:
            raise ValueError(f"{type(self).__name__} inválido:\n  " + "\n  ".join(errors))
        if original:
            original(self, *args)

    cls.__post_init__ = __post_init__
    return cls

@dataclass(frozen=True, slots=True)
@validated
class Reading:
    sensor: str = check(max_len=10)
    celsius: float = check(ge=-40, le=85)
    unit: str = check(choices=("C", "F"), default="C")

print(Reading("s-01", 21.5))
try:
    Reading("sensor-muito-longo", 120, "K")
except ValueError as e:
    print(e)
try:
    Reading(42, "quente")
except ValueError as e:
    print(e)

Reading(sensor='s-01', celsius=21.5, unit='C')
Reading inválido:
  sensor: tamanho máximo 10
  celsius: deve ser <= 85
  unit: deve ser um de ('C', 'F')
Reading inválido:
  sensor: esperado <class 'str'>, recebido int
  celsius: esperado <class 'float'>, recebido str


> ⚠️ **A ordem dos decoradores importa — e o erro é silencioso.** O `__init__` gerado pelo `@dataclass` só inclui a chamada
> a `self.__post_init__()` se o método **já existir** quando o decorador roda. Se `@validated` ficasse por fora
> (aplicado depois), o `__post_init__` seria anexado à classe mas **nunca chamado**: nenhuma validação, nenhum erro.
> Por isso `@validated` fica por dentro (mais perto do `class`). A prova:

In [6]:
@validated
@dataclass
class WrongOrder:
    x: int = check(ge=0)

print("ordem errada aceitou x=-5 sem reclamar:", WrongOrder(-5))

ordem errada aceitou x=-5 sem reclamar: WrongOrder(x=-5)


## 3. `make_dataclass`: classes geradas em runtime

Quando a forma dos dados só é conhecida em execução — cabeçalho de CSV, schema vindo de banco/planilha, colunas
selecionadas pelo usuário — dá para gerar a classe dinamicamente.

In [7]:
import csv, io

CSV = """sku,description,qty,price
A1,Cabo USB-C,10,19.9
B2,Hub 4 portas,3,149.0
"""

def dataclass_from_csv(name: str, text: str, types_hint: dict[str, type]):
    reader = csv.DictReader(io.StringIO(text))
    spec = [(col, types_hint.get(col, str)) for col in reader.fieldnames]
    cls = make_dataclass(name, spec, frozen=True, slots=True)
    rows = [cls(**{c: t(r[c]) for c, t in spec}) for r in reader]
    return cls, rows

Row, rows = dataclass_from_csv("InventoryRow", CSV, {"qty": int, "price": float})
print(Row, Row.__slots__)
print(rows)
print("total em estoque:", sum(r.qty * r.price for r in rows))

<class '__main__.InventoryRow'> ('sku', 'description', 'qty', 'price')
[InventoryRow(sku='A1', description='Cabo USB-C', qty=10, price=19.9), InventoryRow(sku='B2', description='Hub 4 portas', qty=3, price=149.0)]
total em estoque: 646.0


## 4. Dataclasses genéricas: `Result[T]`

Serviços de domínio podem devolver sucesso **ou** erro sem usar exceção como controle de fluxo.
Com a sintaxe de genéricos do Python 3.12 (PEP 695) e `match/case`, isso fica legível e tipado.

In [8]:
@dataclass(frozen=True, slots=True)
class Ok[T]:
    value: T

@dataclass(frozen=True, slots=True)
class Err:
    code: str
    message: str

type Result[T] = Ok[T] | Err

@dataclass(frozen=True, slots=True)
class Quote:
    sku: str
    total: Decimal

def quote(sku: str, qty: int, stock: dict[str, tuple[int, Decimal]]) -> Result[Quote]:
    if sku not in stock:
        return Err("not_found", f"{sku} não existe")
    available, price = stock[sku]
    if qty > available:
        return Err("insufficient_stock", f"pedido {qty}, disponível {available}")
    return Ok(Quote(sku, price * qty))

STOCK = {"A1": (10, Decimal("19.90")), "B2": (3, Decimal("149.00"))}
for sku, qty in [("A1", 2), ("B2", 5), ("Z9", 1)]:
    match quote(sku, qty, STOCK):
        case Ok(Quote(total=total)):
            print(f"{sku}: ok, total {total}")
        case Err(code, msg):
            print(f"{sku}: erro {code} -> {msg}")

A1: ok, total 39.80
B2: erro insufficient_stock -> pedido 5, disponível 3
Z9: erro not_found -> Z9 não existe


## 5. Seu próprio decorador com `dataclass_transform` (PEP 681)

Equipes costumam repetir `@dataclass(frozen=True, slots=True, kw_only=True)` em todo lugar. Um decorador próprio
padroniza isso — e `typing.dataclass_transform` avisa ao mypy/pyright/IDE que ele se comporta como dataclass
(autocomplete e checagem do `__init__` continuam funcionando). É o mesmo mecanismo que Pydantic, attrs e SQLAlchemy usam.

In [9]:
from typing import dataclass_transform

ENTITY_REGISTRY: dict[str, type] = {}

@dataclass_transform(frozen_default=True, kw_only_default=True)
def entity(cls=None, /, *, table: str | None = None):
    def wrap(c):
        dc = dataclass(frozen=True, slots=True, kw_only=True)(c)
        dc.__table__ = table or c.__name__.lower() + "s"
        ENTITY_REGISTRY[dc.__table__] = dc
        return dc
    return wrap if cls is None else wrap(cls)

@entity
class Customer:
    id: int
    name: str
    email: str

@entity(table="sales_orders")
class SalesOrder:
    id: int
    customer_id: int
    total: Decimal = Decimal("0")

print(Customer(id=1, name="Ana", email="ana@x.com"))
print({k: v.__name__ for k, v in ENTITY_REGISTRY.items()})
try:
    Customer(1, "Ana", "ana@x.com")
except TypeError as e:
    print("kw_only padrão:", e)

Customer(id=1, name='Ana', email='ana@x.com')
{'customers': 'Customer', 'sales_orders': 'SalesOrder'}
kw_only padrão: Customer.__init__() takes 1 positional argument but 4 were given


## 6. `cached_property`, `slots` e `frozen`

`functools.cached_property` guarda o resultado no `__dict__` da instância. Daí:

| combinação | funciona? | por quê |
|---|---|---|
| dataclass comum | ✅ | tem `__dict__` |
| `frozen=True` | ✅ | `cached_property` escreve direto no `__dict__`, sem passar pelo `__setattr__` bloqueado |
| `slots=True` | ❌ | não existe `__dict__` |

In [10]:
from functools import cached_property

@dataclass(frozen=True)
class Polygon:
    points: tuple[tuple[float, float], ...]

    @cached_property
    def area(self) -> float:
        print("  (calculando área...)")
        pts = self.points
        return abs(sum(x1 * y2 - x2 * y1 for (x1, y1), (x2, y2) in zip(pts, pts[1:] + pts[:1]))) / 2

sq = Polygon(((0, 0), (4, 0), (4, 4), (0, 4)))
print(sq.area, sq.area)

@dataclass(slots=True)
class PolygonSlots:
    points: tuple
    @cached_property
    def area(self) -> float:
        return 0.0

try:
    PolygonSlots(()).area
except TypeError as e:
    print("TypeError:", e)

  (calculando área...)
16.0 16.0
TypeError: No '__dict__' attribute on 'PolygonSlots' instance to cache 'area' property.


Alternativas com `slots=True`: calcular no `__post_init__` para um campo `init=False`, ou declarar um slot de cache manual.

In [11]:
@dataclass(slots=True)
class PolygonFast:
    points: tuple[tuple[float, float], ...]
    area: float = field(init=False, repr=False)

    def __post_init__(self):
        pts = self.points
        self.area = abs(sum(x1 * y2 - x2 * y1 for (x1, y1), (x2, y2) in zip(pts, pts[1:] + pts[:1]))) / 2

print(PolygonFast(((0, 0), (4, 0), (4, 4), (0, 4))).area)

16.0


## 7. `copy.replace` e `__replace__` (Python 3.13)

O Python 3.13 generalizou o `dataclasses.replace` em `copy.replace`, que funciona com qualquer classe que implemente
`__replace__` — dataclasses, `namedtuple`, `datetime`... Útil para APIs imutáveis uniformes.

In [12]:
import copy
from collections import namedtuple

@dataclass(frozen=True)
class Money:
    amount: Decimal
    currency: str = "BRL"

P = namedtuple("P", "x y")
print(copy.replace(Money(Decimal("10")), amount=Decimal("20")))
print(copy.replace(P(1, 2), y=99))
print(copy.replace(datetime(2026, 1, 1), month=11, day=10))

Money(amount=Decimal('20'), currency='BRL')
P(x=1, y=99)
2026-11-10 00:00:00


## 8. Benchmark: memória e tempo de criação

200 mil objetos com 4 campos. Medimos com `tracemalloc` (memória alocada de verdade) e `timeit`.
Os valores absolutos variam por máquina; as **proporções** são o que importa.

In [13]:
from collections import namedtuple
from pydantic import BaseModel

class PlainClass:
    def __init__(self, a, b, c, d):
        self.a, self.b, self.c, self.d = a, b, c, d

class PlainSlots:
    __slots__ = ("a", "b", "c", "d")
    def __init__(self, a, b, c, d):
        self.a, self.b, self.c, self.d = a, b, c, d

@dataclass
class DC:
    a: int; b: float; c: str; d: bool

@dataclass(slots=True)
class DCSlots:
    a: int; b: float; c: str; d: bool

@dataclass(frozen=True, slots=True)
class DCFrozenSlots:
    a: int; b: float; c: str; d: bool

NT = namedtuple("NT", "a b c d")

class PM(BaseModel):
    a: int; b: float; c: str; d: bool

makers = {
    "dict": lambda i: {"a": i, "b": 1.5, "c": "x", "d": True},
    "classe comum": lambda i: PlainClass(i, 1.5, "x", True),
    "classe com __slots__": lambda i: PlainSlots(i, 1.5, "x", True),
    "namedtuple": lambda i: NT(i, 1.5, "x", True),
    "@dataclass": lambda i: DC(i, 1.5, "x", True),
    "@dataclass(slots=True)": lambda i: DCSlots(i, 1.5, "x", True),
    "@dataclass(frozen, slots)": lambda i: DCFrozenSlots(i, 1.5, "x", True),
    "pydantic BaseModel": lambda i: PM(a=i, b=1.5, c="x", d=True),
}

N = 200_000
rows = []
for name, mk in makers.items():
    tracemalloc.start()
    objs = [mk(i) for i in range(N)]
    mem = tracemalloc.get_traced_memory()[0]
    tracemalloc.stop()
    del objs
    t = min(timeit.repeat(lambda: mk(1), number=50_000, repeat=3)) / 50_000 * 1e9
    rows.append((name, mem / N, t))

base_mem = rows[0][1]
print(f"{'estrutura':<28}{'bytes/obj':>10}{'vs dict':>9}{'ns/criação':>12}")
for name, m, t in rows:
    print(f"{name:<28}{m:>10.0f}{m / base_mem:>8.2f}x{t:>12.0f}")

estrutura                    bytes/obj  vs dict  ns/criação
dict                               224    1.00x          89
classe comum                       144    0.64x         170
classe com __slots__               104    0.46x         165
namedtuple                         120    0.54x         194
@dataclass                         144    0.64x         163
@dataclass(slots=True)             104    0.46x         144
@dataclass(frozen, slots)          104    0.46x         419
pydantic BaseModel                 512    2.29x         747


O que esses números costumam mostrar:

* `slots=True` corta a memória por objeto de forma significativa em relação à dataclass comum (que paga um `__dict__` por instância).
  Em 10 milhões de objetos, isso é a diferença entre caber ou não na RAM.
* `frozen=True` deixa a **criação** mais lenta (o `__init__` usa `object.__setattr__`), mas não muda a memória.
* Pydantic custa mais por objeto — o preço da validação. Na borda, irrelevante; num loop interno de milhões, relevante.

## 9. Ponte com Pydantic: validar para dentro de uma dataclass stdlib

`TypeAdapter` aceita **dataclasses da stdlib**: a classe continua 100% stdlib (o domínio não importa Pydantic),
mas na borda você ganha validação, conversão, erros completos e JSON Schema.

In [14]:
from pydantic import TypeAdapter, ValidationError

OrderAdapter = TypeAdapter(Order)           # a MESMA Order da seção 1

o = OrderAdapter.validate_python(raw)
print(type(o).__module__, type(o).__name__, "| é a dataclass stdlib:", type(o) is Order)
print(o.items[0])

try:
    OrderAdapter.validate_python({**raw, "status": "cancelled", "items": [{"sku": "A1", "qty": "um", "unit_price": "x"}]})
except ValidationError as e:
    for err in e.errors(include_url=False):
        print(f"  {'.'.join(map(str, err['loc'])):<24} {err['type']}")

print(OrderAdapter.dump_json(o)[:120], b"...")
print(list(OrderAdapter.json_schema()["properties"]))

__main__ Order | é a dataclass stdlib: True
Item(sku='A1', qty=2, unit_price=Decimal('19.90'))
  status                   enum
  items.0.qty              int_parsing
  items.0.unit_price       decimal_parsing
b'{"id":"0b8e6c6e-52a5-4b8f-9d1e-3c1f8a7e2b10","status":"paid","created_at":"2026-11-10T09:30:00Z","items":[{"sku":"A1","q' b'...'
['id', 'status', 'created_at', 'items', 'discount', 'metadata']


Compare com a seção 1: o Pydantic reportou **todos** os erros de uma vez, com caminho, e ainda serializa e gera schema.
Também existe `pydantic.dataclasses.dataclass` — uma dataclass que valida no `__init__` — útil quando você quer validação
sempre ligada sem migrar para `BaseModel`.

In [15]:
from pydantic.dataclasses import dataclass as pydantic_dataclass
from pydantic import Field

@pydantic_dataclass(frozen=True)
class Temperature:
    sensor: str
    celsius: float = Field(ge=-40, le=85)

print(Temperature("s-01", "21.5"))
try:
    Temperature("s-01", 120)
except ValidationError as e:
    print(e.errors(include_url=False)[0]["msg"])

Temperature(sensor='s-01', celsius=21.5)
Input should be less than or equal to 85


## 10. Arquitetura de bordas: DTO Pydantic ↔ domínio dataclass

```
HTTP JSON ──► OrderIn (Pydantic) ──to_domain()──► Order (dataclass) ──► serviço ──► OrderSummary (dataclass)
                valida 1x                          regra de negócio                    │
HTTP JSON ◄── OrderOut (Pydantic) ◄──from_attributes────────────────────────────────────┘
```

O domínio **não importa Pydantic**: dá para testá-lo, reutilizá-lo num worker, CLI ou notebook sem tocar na camada HTTP.

In [16]:
from pydantic import BaseModel, ConfigDict, Field as PField
from pydantic.alias_generators import to_camel

# ---------- domínio (stdlib pura) ----------
@dataclass(frozen=True, slots=True)
class Line:
    sku: str
    qty: int
    unit_price: Decimal

    @property
    def subtotal(self) -> Decimal:
        return self.unit_price * self.qty

@dataclass(frozen=True, slots=True)
class Cart:
    customer_id: int
    lines: tuple[Line, ...]

@dataclass(frozen=True, slots=True)
class CartSummary:
    customer_id: int
    items: int
    gross: Decimal
    discount: Decimal
    net: Decimal

def summarize(cart: Cart, loyalty_pct: Decimal) -> CartSummary:
    gross = sum((l.subtotal for l in cart.lines), Decimal("0"))
    discount = (gross * loyalty_pct / 100).quantize(Decimal("0.01"))
    return CartSummary(cart.customer_id, sum(l.qty for l in cart.lines), gross, discount, gross - discount)

# ---------- borda (Pydantic) ----------
class Api(BaseModel):
    model_config = ConfigDict(alias_generator=to_camel, validate_by_name=True, serialize_by_alias=True)

class LineIn(Api):
    model_config = ConfigDict(extra="forbid")
    sku: str = PField(min_length=1)
    qty: int = PField(ge=1, le=100)
    unit_price: Decimal = PField(gt=0, decimal_places=2)

class CartIn(Api):
    model_config = ConfigDict(extra="forbid")
    customer_id: int
    lines: list[LineIn] = PField(min_length=1)

    def to_domain(self) -> Cart:
        return Cart(self.customer_id, tuple(Line(l.sku, l.qty, l.unit_price) for l in self.lines))

class CartSummaryOut(Api):
    model_config = ConfigDict(from_attributes=True)
    customer_id: int
    items: int
    gross: Decimal
    discount: Decimal
    net: Decimal

# ---------- "handler" ----------
def post_cart_summary(body: str) -> str:
    cart_in = CartIn.model_validate_json(body)           # borda: valida
    summary = summarize(cart_in.to_domain(), Decimal("5"))  # núcleo: regra pura
    return CartSummaryOut.model_validate(summary).model_dump_json()   # borda: serializa

print(post_cart_summary('{"customerId": 7, "lines": [{"sku": "A1", "qty": 2, "unitPrice": "19.90"}, {"sku": "B2", "qty": 1, "unitPrice": "149.00"}]}'))

{"customerId":7,"items":3,"gross":"188.80","discount":"9.44","net":"179.36"}


## 11. Checklist

| ✔ | prática |
|---|---|
| ☐ | Defaults mutáveis sempre com `default_factory` |
| ☐ | Value objects e configs: `frozen=True`; coleções internas como `tuple`/`frozenset` |
| ☐ | Muitos objetos: `slots=True` (e lembre que `cached_property` deixa de funcionar) |
| ☐ | Construtores com muitos campos ou herança com defaults: `kw_only=True` / `KW_ONLY` |
| ☐ | Validação mínima no `__post_init__`; validação de dado externo **na borda** (Pydantic) |
| ☐ | Padronize opções com um decorador próprio + `dataclass_transform` |
| ☐ | `asdict` é deep copy: evite em caminhos quentes com estruturas grandes |
| ☐ | Domínio sem dependência de Pydantic; `TypeAdapter(MinhaDataclass)` na borda quando quiser validar direto nela |

### Exercícios
1. Faça o `load()` da seção 1 coletar **todos** os erros antes de levantar.
2. Estenda `@validated` com a regra `pattern` (regex).
3. Gere, com `make_dataclass`, uma classe a partir de um JSON Schema simples (`properties` + `type`).
4. Meça quanto o `frozen=True` custa na criação comparado a `slots=True` sozinho na sua máquina.

➡️ Próximo módulo: [`../enum/enum.md`](../enum/enum.md)